# Experiment 14 – Object Detection Using CNN

### Additional Experiment – TensorFlow/Keras

**Aim:** To implement a simple object detection system using a Convolutional Neural Network (CNN) and a pre-trained object detection model in TensorFlow/Keras.

### Learning Objectives
- Understand the difference between image classification and object detection.
- Understand bounding boxes and confidence scores.
- Use a pre-trained CNN-based object detection model.
- Detect multiple objects in a given image.
- Display detected objects with bounding boxes.

## 1. Theory

**Image classification** answers: *What is present in the image?*

**Object detection** answers: *What objects are present and where are they located?*

An object detector produces:

- **Class label** – the name of the detected object.
- **Confidence score** – how confident the model is about the detection.
- **Bounding box** – the location of the object in the image.

### Basic Flow

```text
Input Image
     ↓
Pre-trained CNN Detector
     ↓
Feature Extraction
     ↓
Object Detection
     ↓
Class + Confidence + Bounding Box
     ↓
Detected Image
```

**Note:** For a practical Colab experiment, we use a pre-trained **SSD MobileNet V2** detector. It is CNN-based, lightweight, and suitable for demonstrating object detection without training a detector from scratch.

In [ ]:
# Step 1: Install TensorFlow Hub
!pip -q install tensorflow-hub

In [ ]:
# Step 2: Import required libraries
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

print("TensorFlow version:", tf.__version__)
print("TensorFlow Hub loaded successfully.")

## 2. Load a Pre-trained CNN Object Detection Model

We use **SSD MobileNet V2** from TensorFlow Hub.

The model has already learned to detect common objects. This allows students to focus on understanding object detection rather than spending a long time training a complex detector.

In [ ]:
# Step 3: Load the pre-trained SSD MobileNet V2 detector
MODEL_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"

print("Loading model...")
detector = hub.load(MODEL_URL)
print("Object detection model loaded successfully.")

## 3. Upload an Image

Choose an image containing one or more common objects, such as people, cars, bicycles, dogs, cats, bottles, chairs, or other everyday objects.

In [ ]:
# Step 4: Upload an image
uploaded = files.upload()
image_path = next(iter(uploaded.keys()))

print("Selected image:", image_path)

In [ ]:
# Step 5: Read and display the input image
image = Image.open(image_path).convert("RGB")
image_np = np.array(image)

plt.figure(figsize=(10, 7))
plt.imshow(image_np)
plt.axis("off")
plt.title("Input Image")
plt.show()

print("Image size:", image.size)

## 4. Prepare the Image for the Model

The detector expects an image tensor. We add a batch dimension so that one image can be passed to the model.

In [ ]:
# Step 6: Convert image into a TensorFlow input tensor
input_tensor = tf.convert_to_tensor(image_np, dtype=tf.uint8)
input_tensor = input_tensor[tf.newaxis, ...]

print("Input tensor shape:", input_tensor.shape)

## 5. Perform Object Detection

The model returns bounding boxes, class IDs, confidence scores, and the number of detections.

In [ ]:
# Step 7: Run the detector
result = detector(input_tensor)

# Convert TensorFlow tensors to NumPy arrays
result = {key: value.numpy() for key, value in result.items()}

print("Available outputs:", list(result.keys()))

## 6. Class Labels

The detector uses COCO object categories. The following list provides the common labels used by the model.

In [ ]:
# Step 8: COCO class labels
COCO_LABELS = {
    1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane",
    6: "bus", 7: "train", 8: "truck", 9: "boat", 10: "traffic light",
    11: "fire hydrant", 13: "stop sign", 14: "parking meter", 15: "bench",
    16: "bird", 17: "cat", 18: "dog", 19: "horse", 20: "sheep",
    21: "cow", 22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe",
    27: "backpack", 28: "umbrella", 31: "handbag", 32: "tie",
    33: "suitcase", 34: "frisbee", 35: "skis", 36: "snowboard",
    37: "sports ball", 38: "kite", 39: "baseball bat", 40: "baseball glove",
    41: "skateboard", 42: "surfboard", 43: "tennis racket", 44: "bottle",
    46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon",
    51: "bowl", 52: "banana", 53: "apple", 54: "sandwich", 55: "orange",
    56: "broccoli", 57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut",
    61: "cake", 62: "chair", 63: "couch", 64: "potted plant", 65: "bed",
    67: "dining table", 70: "toilet", 72: "tv", 73: "laptop", 74: "mouse",
    75: "remote", 76: "keyboard", 77: "cell phone", 78: "microwave",
    79: "oven", 80: "toaster", 81: "sink", 82: "refrigerator",
    84: "book", 85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear",
    89: "hair drier", 90: "toothbrush"
}

print("Number of class labels:", len(COCO_LABELS))

In [ ]:
# Step 9: Display detections above the confidence threshold
THRESHOLD = 0.40

boxes = result["detection_boxes"][0]
scores = result["detection_scores"][0]
classes = result["detection_classes"][0].astype(int)

print("Detected objects:\n")

for i, score in enumerate(scores):
    if score >= THRESHOLD:
        class_id = classes[i]
        label = COCO_LABELS.get(class_id, f"class_{class_id}")
        print(f"{label:20s} | Confidence: {score * 100:.2f}%")

## 7. Draw Bounding Boxes

A bounding box is represented by four normalized values:

```text
[ymin, xmin, ymax, xmax]
```

These values are converted into pixel coordinates and drawn around detected objects.

In [ ]:
# Step 10: Draw bounding boxes on the image
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image_np)

height, width, _ = image_np.shape

for i, score in enumerate(scores):
    if score >= THRESHOLD:
        ymin, xmin, ymax, xmax = boxes[i]
        left = xmin * width
        top = ymin * height
        box_width = (xmax - xmin) * width
        box_height = (ymax - ymin) * height

        rect = plt.Rectangle(
            (left, top), box_width, box_height,
            fill=False, linewidth=2
        )
        ax.add_patch(rect)

        class_id = classes[i]
        label = COCO_LABELS.get(class_id, f"class_{class_id}")
        text = f"{label}: {score * 100:.1f}%"
        ax.text(left, max(0, top - 5), text, fontsize=10,
                bbox=dict(facecolor="white", alpha=0.8))

ax.axis("off")
ax.set_title("Object Detection Results")
plt.show()

## 8. Change the Confidence Threshold

The confidence threshold controls which detections are displayed.

Try values such as:

- `0.30` → more detections, but possibly more incorrect detections.
- `0.50` → fewer, more confident detections.
- `0.70` → only high-confidence detections.

Students should compare the results.

In [ ]:
# Step 11: Try a different confidence threshold
THRESHOLD = 0.60

print("Detections with confidence >=", THRESHOLD)
for i, score in enumerate(scores):
    if score >= THRESHOLD:
        class_id = classes[i]
        label = COCO_LABELS.get(class_id, f"class_{class_id}")
        print(f"{label:20s} | {score * 100:.2f}%")

## 9. Classification vs Object Detection

| Feature | Image Classification | Object Detection |
|---|---|---|
| Output | Class label | Class + location |
| Number of objects | Usually one main prediction | Multiple objects possible |
| Bounding box | No | Yes |
| Example | “This is a dog” | “Dog is here in this region” |

Object detection is useful in applications such as autonomous vehicles, surveillance, robotics, traffic monitoring, and smart retail.

## 10. Student Practice

1. Upload an image containing at least three objects.
2. Record the detected objects and confidence scores.
3. Change the threshold from `0.40` to `0.60`.
4. Compare the number of detections at both thresholds.
5. Try an image containing a person and a vehicle.
6. Explain what a bounding box represents.
7. Explain why confidence thresholds are used.

### Observation Table

| Image | Detected Object | Confidence | Correct? |
|---|---|---:|---|
| Image 1 | __________ | ______ % | Yes / No |
| Image 1 | __________ | ______ % | Yes / No |
| Image 2 | __________ | ______ % | Yes / No |
| Image 2 | __________ | ______ % | Yes / No |

## 11. Result

Thus, object detection was successfully implemented using a pre-trained **SSD MobileNet V2 CNN-based detector** in TensorFlow. The system detected objects in a given image and displayed their class labels, confidence scores, and bounding boxes.

## Viva Questions

1. What is object detection?
2. How is object detection different from image classification?
3. What is a bounding box?
4. What is a confidence score?
5. What is CNN?
6. What is SSD?
7. What is MobileNet?
8. Why do we use a pre-trained model?
9. What happens when the confidence threshold is increased?
10. Give two real-world applications of object detection.

## 12. Conclusion

In this experiment, students learned the basic principles of object detection and implemented a practical CNN-based object detector using a pre-trained SSD MobileNet V2 model. The experiment demonstrated how a deep-learning model can identify multiple objects and locate them using bounding boxes.